Pseudocode

Due to this input system attempting to be easier to type the sequences only go on one side instead of having a corresponding opening and closing pair. This means that instead of being able to just read forward I will also need to read backwards for some sequences to support nesting them. This may have been a stupid design decision, but it's one that I made a long time ago and have lots of data using.

1. The parser should have modes to determine what direction to look for. The modes would be defined by the sequence.
    1. Default: Read all forward characters until the next sequence.
        - Should not return to any mode as it should only occur at the entry character or the end of another sequence.
    2. Forward: After a sequence take all forward characters until the next sequence.
        - Returns to Default mode (if it's a Backward sequence) or to Forward mode (if it's a Forward sequence).
        - Trim the results for leading and trailing spaces.
    3. Backward: After a sequence take all characters from the previous Default mode.
        - Returns to Default mode (if it's a Backward sequence) or to Forward mode (if it's a Forward sequence).
        - Trim the results for leading and trailing spaces.
2. Support for nested structures, specifically only comments so far.
    1. If the current mode is Default mode and you encounter a Forward mode then create a child parser to store as a child of the previous Default mode.
3. Handling empty sequences
    - If a Default mode encounters a Forward mode add it as a child as above.
    - When the Default mode is returned if there are no characters after then discard the Default mode and replace it with the child Forward mode. 
    - If a default mode reaches the end without a sequence then discard it.

Wait actually I don't think I even ever need to read left with the system I have. I always read to the right, it's just a matter of handling the nested values and the modes.

Parsing Current Parent

In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod

class ParseCurrent(ABC):
    """
    """
    def __init__(self, source_text: str, parsing_mode: str):
        self.processed_text: str = ""
        self.unprocessed_text: str = source_text
        self.parsing_mode: str = parsing_mode
        self.child: list[ParseCurrent] = []

    @abstractmethod
    def test(self):
        """
        """
        pass

In [ ]:
from __future__ import annotations
class ParseCurrent:
    """
    """
    def __init__(self, source_text: str):
        self.processed_text: str = ""
        self.unprocessed_text: str = source_text
        

    def getNextChars(self, next_character: int = 1) -> str:
        """
        """
        if next_character >= 1 and len(self.unprocessed_text) >= next_character + 1:
            return self.unprocessed_text[next_character]
        raise IndexError("ParseCurrent.getNextChar() there are not enough characters in unprocessed_text")

    def getCurChar(self) -> str:
        """
        """
        if len(self.unprocessed_text) >= 1:
            return self.unprocessed_text[0]
        raise IndexError("ParseCurrent.getNextChar() there are not enough characters in unprocessed_text")
    
    def getPrevChars(self, previous_character: int = -1) -> str:
        """
        """
        if previous_character <= -1 and len(self.processed_text) >=  -previous_character:
            return self.processed_text[len(self.processed_text) + previous_character]
        raise IndexError("ParseCurrent.getNextChar() there are not enough characters in processed")
    
    def itrChars(self, itr_characters: int = 1) -> None:
        """
        """
        if len(self.unprocessed_text) >= itr_characters:
            self.processed_text = "".join((self.processed_text, self.unprocessed_text[0:itr_characters]))
            self.unprocessed_text = self.unprocessed_text[itr_characters:]
            # Probably need something here to check if there's no more unprocessed characters, possibly another error
        else:
            raise IndexError("ParseCurrent.itrChar() there are no more unprocessed characters")
        
    def getChars(self, itr_characters: int = 0):
        # No match/switch statement in this python version :(
        if itr_characters == 0:
            return self.getCurChar()
        if itr_characters >= 1:
            return self.getNextChars(itr_characters)
        else:
            return self.getPrevChars(itr_characters)

    def processMatch(self, match_chars: int) -> tuple[str, ParseCurrent]:
        """
        """
        return (self.processed_text, ParseCurrent(self.unprocessed_text[match_chars:]))
        